# TERZ net comparison: HDBSCAN sampling in 3D (ipyvolume)

Same pipeline as `net_compare_density_sampling_hdbscan.ipynb`, but every embedding has **3 components** and every plot is an interactive **ipyvolume** figure (rotate, zoom, tick boxes to show or hide groups):

1. loads the TERZ parquet, normalises each trigger by its max and standardises the features (as before),
2. computes a **3D t-SNE** and a **3D UMAP** (cached on disk, the 3D files have their own names, so the 2D caches are not touched),
3. runs **HDBSCAN on the 3D embedding** of `LABEL_FROM` (cached) and draws `PER_CLUSTER` random triggers from every cluster with at least `MIN_CLUSTER_SIZE` triggers,
4. **figure 1** (per projection): the full 3D embedding, with the selected triggers drawn large and in the colour of their cluster, the other clustered triggers small, and the noise grey. Tick boxes show or hide each group,
5. computes t-SNE and UMAP (3D) **again on the subset only**, **figure 2**: the subset coloured by the first-round cluster, one tick box per cluster.

The 2x2 panel figure (KDE density, k-means), the thumbnail map and `LABEL_METHOD` are not part of this notebook: a KDE surface needs a 4th dimension in 3D, and the thumbnails are a 2D image map.

Every figure is also saved as a standalone HTML file in `FIG_DIR` (the tick boxes are not saved, only the 3D scene).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipyvolume as ipv
import ipywidgets as widgets
from IPython.display import display
from matplotlib.colors import to_hex
from sklearn.cluster import HDBSCAN
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
METRIC = "euclidean"
N_COMPONENTS = 3                          # the plots below need exactly 3 (sklearn's Barnes-Hut t-SNE supports up to 3)

PARQUET_DIR = Path("year_TERZ_parquets")
CACHE_DIR = Path("TERZ_embeddings_cache")      # embeddings are stored here
CACHE_DIR.mkdir(exist_ok=True)
FIG_DIR = Path("TERZ_figures_net_compare_sampling_3d")
SAVE_HTML = True                          # True -> also save every ipyvolume figure as a standalone HTML file in FIG_DIR

# What to run.
YEARS = [2022]                            # e.g. [2022, 2023, 2024, 2025]
ONTS = [25] #[15, 20, 25]
HW = [6, 12] #[1, 2, 4, 6, 12]

# t-SNE on the full set (same values as before)
PERPLEXITY = 350
EARLY_EXAGGERATION = 14

# UMAP on the full set
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1

# HDBSCAN on the 3D embedding. The scale of the 3D coordinates is not the one of the 2D embedding,
# so min_samples may need a new value (use CLUSTER_OVERRIDES per dataset).
CLUSTER_PARAMS = {
    "tsne": {"hdbscan_min_samples": 500},
    "umap": {"hdbscan_min_samples": 500},
}
# Per-dataset overrides, keyed by (year, ont, projection), e.g.
# CLUSTER_OVERRIDES = {(2022, 15, "umap"): {"hdbscan_min_samples": 200}}
CLUSTER_OVERRIDES = {}

PROJECTIONS = {"tsne": "t-SNE", "umap": "UMAP"}

# Balanced subset + second round of dimensionality reduction
LABEL_FROM = "tsne"                       # projection whose HDBSCAN clusters are the labels: "tsne" or "umap"
INCLUDE_NOISE = False                     # True -> the noise points (label -1) are sampled as one more group
PER_CLUSTER = 750                         # triggers drawn at random from every cluster
MIN_CLUSTER_SIZE = PER_CLUSTER            # clusters with fewer triggers are discarded
SUBSET_DIR = Path("TERZ_subsets")         # selected row indices + labels are stored here
SUBSET_DIR.mkdir(exist_ok=True)
# embedding parameters for the subset
SUB_PERPLEXITY = 15
SUB_EARLY_EXAGGERATION = EARLY_EXAGGERATION
SUB_UMAP_N_NEIGHBORS = UMAP_N_NEIGHBORS
SUB_UMAP_MIN_DIST = UMAP_MIN_DIST

# ipyvolume look. Sizes are in ipyvolume units (the scene is scaled to [-1, 1]); if the points look too small or too
# big on your screen change these. "circle_2d" is much lighter than "sphere" when there are tens of thousands of points.
FIG_WIDTH, FIG_HEIGHT = 900, 750          # pixels
FULL_MARKER = "circle_2d"                 # marker of the full-embedding figure
SIZE_SELECTED = 3.0                       # selected triggers
SIZE_UNSELECTED = 1.0                     # clustered triggers that were not selected
SIZE_NOISE = 0.8                          # noise triggers that were not selected
SUB_MARKER = "sphere"                     # marker of the subset figure (only a few thousand points)
SIZE_SUBSET = 1.2

**Clustering parameters.** HDBSCAN now runs on the 3D coordinates, so the clusters can differ from the 2D ones (the 2D labels are not reused). `hdbscan_min_samples` is the same value as in the 2D notebook; if you get all noise, or one giant cluster, change it in `CLUSTER_PARAMS` (or per dataset in `CLUSTER_OVERRIDES`). The HDBSCAN labels are cached next to the embedding, so changing only `PER_CLUSTER` or the plot settings never recomputes them.

The first 3D t-SNE of the whole set with `PERPLEXITY = 350` is considerably slower than the 2D one; it is cached, so it only runs once.

In [2]:
def feature_matrix(year, ont, hw, rows=None):
    """Load one parquet, normalise each row by its max and standardise.
    If `rows` is given only those rows are kept, and the standardisation is fitted on them."""
    path = PARQUET_DIR / f"TERZ_HHZ_hw{hw}_ont{ont}_{year}.parquet"
    df = pd.read_parquet(path)

    feature_columns = [col for col in df.columns if col.startswith("feature_")]
    if len(feature_columns) == 0:
        raise ValueError(f"No feature_* columns found in {path}.")

    X = df[feature_columns].to_numpy(dtype="float64")
    if rows is not None:
        X = X[rows]
    print(f"{year} ont={ont}: {X.shape[0]} triggers, {X.shape[1]} features")

    X_norm = X / X.max(axis=1, keepdims=True)
    return StandardScaler().fit_transform(X_norm)


def compute_tsne(X_scaled):
    return TSNE(n_components=N_COMPONENTS, perplexity=PERPLEXITY, early_exaggeration=EARLY_EXAGGERATION,
                init="pca", learning_rate="auto", random_state=RANDOM_STATE,
                metric=METRIC).fit_transform(X_scaled)


def compute_umap(X_scaled):
    import umap  # imported here so cached embeddings can be plotted without umap-learn
    return umap.UMAP(n_components=N_COMPONENTS, n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST,
                     metric=METRIC, random_state=RANDOM_STATE).fit_transform(X_scaled)


COMPUTE = {"tsne": compute_tsne, "umap": compute_umap}


def cache_path(year, ont, projection, hw):
    # the parameters (and the number of components) are part of the file name, so a stale embedding is never reused
    if projection == "tsne":
        tag = f"tsne{N_COMPONENTS}d_perp{PERPLEXITY}_ee{EARLY_EXAGGERATION}"
    else:
        tag = f"umap{N_COMPONENTS}d_nn{UMAP_N_NEIGHBORS}_md{UMAP_MIN_DIST}"
    return CACHE_DIR / f"TERZ_hw{hw}_ont{ont}_{year}_{tag}.npy"


def get_embeddings(year, ont, hw):
    """Return {"tsne": (n, 3), "umap": (n, 3)}; computes and caches whatever is missing."""
    paths = {proj: cache_path(year, ont, proj, hw) for proj in COMPUTE}
    X_scaled = feature_matrix(year, ont, hw) if not all(p.exists() for p in paths.values()) else None

    embeddings = {}
    for proj, path in paths.items():
        if path.exists():
            embeddings[proj] = np.load(path)
            print(f"{year} ont={ont}: {PROJECTIONS[proj]} loaded from {path.name}")
        else:
            print(f"{year} ont={ont}: computing {PROJECTIONS[proj]} ({N_COMPONENTS}D) ...")
            embeddings[proj] = COMPUTE[proj](X_scaled)
            np.save(path, embeddings[proj])
    return embeddings


def hdbscan_labels(E, params, year, ont, hw, proj):
    """HDBSCAN labels of one 3D embedding (-1 = noise), cached next to the embedding."""
    emb = cache_path(year, ont, proj, hw)
    path = emb.with_name(f"{emb.stem}_hdbscan_ms{params['hdbscan_min_samples']}.npy")
    if path.exists():
        labels = np.load(path)
        if len(labels) == len(E):
            return labels
    labels = HDBSCAN(min_samples=params["hdbscan_min_samples"], metric=METRIC, copy=True).fit(E).labels_
    np.save(path, labels)
    return labels

## Balanced subset (HDBSCAN) and second embedding

`PER_CLUSTER` triggers are drawn at random from every HDBSCAN cluster of the `LABEL_FROM` embedding that has at least `MIN_CLUSTER_SIZE` triggers (all of them if the cluster is smaller than `PER_CLUSTER`; smaller clusters than `MIN_CLUSTER_SIZE` are discarded; both are printed). Noise (label -1) is left out unless `INCLUDE_NOISE = True`. The selection is saved in `SUBSET_DIR` (`idx` = row positions in the parquet, `labels` = cluster of each selected row, `labels_full` = cluster of every row).

The subset is then embedded again (3D t-SNE and 3D UMAP) from its own features, normalised and standardised using only the subset.

In [3]:
def sampling_labels(E, proj, year, ont, hw):
    """HDBSCAN label of every row of one 3D embedding (-1 = noise)."""
    params = {**CLUSTER_PARAMS[proj], **CLUSTER_OVERRIDES.get((year, ont, proj), {})}
    labels = hdbscan_labels(E, params, year, ont, hw, proj)
    if not (labels >= 0).any():
        raise ValueError(f"HDBSCAN found no cluster (all noise) for {year} ont={ont} hw={hw}; "
                         f"lower hdbscan_min_samples (now {params['hdbscan_min_samples']}) in CLUSTER_PARAMS / CLUSTER_OVERRIDES.")
    return labels


def balanced_indices(labels, per_cluster, min_size):
    """Sorted row indices: `per_cluster` random rows from every cluster with at least `min_size` triggers (all rows
    if it has fewer than `per_cluster`); smaller clusters are discarded. HDBSCAN noise (label -1) is skipped unless INCLUDE_NOISE."""
    rng = np.random.default_rng(RANDOM_STATE)
    chosen = []
    groups = np.unique(labels)
    if not INCLUDE_NOISE:
        groups = groups[groups >= 0]
    for k in groups:
        members = np.flatnonzero(labels == k)
        if len(members) < min_size:
            print(f"    cluster {k}: only {len(members)} triggers (< {min_size}), discarded")
            continue
        if len(members) < per_cluster:
            print(f"    cluster {k}: only {len(members)} triggers, taking all of them")
        chosen.append(rng.choice(members, size=min(per_cluster, len(members)), replace=False))
    if not chosen:
        raise ValueError(f"Every cluster has fewer than MIN_CLUSTER_SIZE = {min_size} triggers; lower MIN_CLUSTER_SIZE.")
    return np.sort(np.concatenate(chosen))


def get_subset(year, ont, hw):
    """Balanced subset of one dataset -> (row indices, HDBSCAN labels of ALL rows)."""
    E = get_embeddings(year, ont, hw)[LABEL_FROM]
    labels_full = sampling_labels(E, LABEL_FROM, year, ont, hw)
    clusters = np.unique(labels_full[labels_full >= 0])
    print(f"{year} ont={ont} hw={hw}: {len(clusters)} HDBSCAN clusters on the {PROJECTIONS[LABEL_FROM]} {N_COMPONENTS}D embedding, "
          f"sizes {np.bincount(labels_full[labels_full >= 0]).tolist()}, "
          f"{int((labels_full < 0).sum())} noise points{'' if INCLUDE_NOISE else ' (not sampled)'}")
    idx = balanced_indices(labels_full, PER_CLUSTER, MIN_CLUSTER_SIZE)
    np.savez(SUBSET_DIR / f"TERZ_hw{hw}_ont{ont}_{year}_sub{PER_CLUSTER}_{LABEL_FROM}_hdbscan_{N_COMPONENTS}d.npz",
             idx=idx, labels=labels_full[idx], labels_full=labels_full)
    return idx, labels_full


def subset_tsne(X_scaled):
    return TSNE(n_components=N_COMPONENTS, perplexity=SUB_PERPLEXITY, early_exaggeration=SUB_EARLY_EXAGGERATION,
                init="pca", learning_rate="auto", random_state=RANDOM_STATE,
                metric=METRIC).fit_transform(X_scaled)


def subset_umap(X_scaled):
    import umap
    return umap.UMAP(n_components=N_COMPONENTS, n_neighbors=SUB_UMAP_N_NEIGHBORS, min_dist=SUB_UMAP_MIN_DIST,
                     metric=METRIC, random_state=RANDOM_STATE).fit_transform(X_scaled)


SUBSET_COMPUTE = {"tsne": subset_tsne, "umap": subset_umap}


def subset_cache_path(year, ont, hw, projection):
    if projection == "tsne":
        tag = f"tsne{N_COMPONENTS}d_perp{SUB_PERPLEXITY}_ee{SUB_EARLY_EXAGGERATION}"
    else:
        tag = f"umap{N_COMPONENTS}d_nn{SUB_UMAP_N_NEIGHBORS}_md{SUB_UMAP_MIN_DIST}"
    return CACHE_DIR / f"TERZ_hw{hw}_ont{ont}_{year}_sub{PER_CLUSTER}_{LABEL_FROM}_hdbscan_{tag}.npz"


def get_subset_embeddings(year, ont, hw, idx):
    """Return {"tsne": (m, 3), "umap": (m, 3)} for the rows `idx`; computes and caches whatever is missing."""
    paths = {proj: subset_cache_path(year, ont, hw, proj) for proj in SUBSET_COMPUTE}
    embeddings = {}
    for proj, path in paths.items():
        embeddings[proj] = None
        if path.exists():
            with np.load(path) as saved:
                if np.array_equal(saved["idx"], idx):          # the cached embedding belongs to this selection
                    embeddings[proj] = saved["embedding"]
                    print(f"{year} ont={ont} hw={hw}: subset {PROJECTIONS[proj]} loaded from {path.name}")

    missing = [proj for proj, E in embeddings.items() if E is None]
    if missing:
        X_scaled = feature_matrix(year, ont, hw, rows=idx)
        for proj in missing:
            print(f"{year} ont={ont} hw={hw}: computing subset {PROJECTIONS[proj]} ({N_COMPONENTS}D) ...")
            embeddings[proj] = SUBSET_COMPUTE[proj](X_scaled)
            np.savez(paths[proj], embedding=embeddings[proj], idx=idx)
    return embeddings

## ipyvolume helpers

The embeddings are centred and scaled by one common factor into `[-1, 1]^3`, so the three axes keep their proportions. Colours are the first-round HDBSCAN clusters (noise grey), the same in both figures. If the figure does not show up inline (VS Code needs internet access to load the ipyvolume widget scripts), open the saved HTML files in `FIG_DIR` in a browser.

In [4]:
def unit_cube(E):
    """Centre the embedding and scale it with one common factor into [-1, 1]^3."""
    centre = (E.max(axis=0) + E.min(axis=0)) / 2
    half = (E.max(axis=0) - E.min(axis=0)).max() / 2
    return (E - centre) / half


def label_colors(labels):
    """One RGB colour per cluster label (same colours in both figures); noise is grey."""
    clusters = [k for k in np.unique(labels) if k >= 0]
    if len(clusters) <= 20:
        cmap = plt.get_cmap("tab10" if len(clusters) <= 10 else "tab20")
        colors = {k: tuple(cmap(i)[:3]) for i, k in enumerate(clusters)}
    else:
        cmap = plt.get_cmap("gist_ncar")
        colors = {k: tuple(cmap(i / len(clusters))[:3]) for i, k in enumerate(clusters)}
    colors[-1] = (0.5, 0.5, 0.5)
    return colors


def color_array(labels, colors):
    """(n, 3) float array with the colour of every row."""
    rgb = np.empty((len(labels), 3))
    for k in np.unique(labels):
        rgb[labels == k] = colors[k]
    return rgb


def swatch(color):
    return widgets.HTML(f"<span style='display:inline-block;width:12px;height:12px;margin:4px 6px 0 0;"
                        f"border-radius:2px;background:{to_hex(color)}'></span>")


def _set_visible(scatter, value):
    names = scatter.trait_names()
    trait = "visible_markers" if "visible_markers" in names else "visible"
    setattr(scatter, trait, bool(value))


def make_controls(items):
    """items = [(text, colour or None, scatter), ...] -> tick boxes that show/hide every scatter object."""
    boxes, rows = [], []
    for text, color, scatter in items:
        cb = widgets.Checkbox(value=True, description=text, indent=False, layout=widgets.Layout(width="300px"))
        cb.observe(lambda change, sc=scatter: _set_visible(sc, change["new"]), names="value")
        boxes.append(cb)
        rows.append(widgets.HBox([swatch(color), cb]) if color is not None else cb)

    def set_all(value):
        for cb in boxes:
            cb.value = value

    show_all, hide_all = widgets.Button(description="show all"), widgets.Button(description="hide all")
    show_all.on_click(lambda _: set_all(True))
    hide_all.on_click(lambda _: set_all(False))
    return widgets.VBox([widgets.HBox([show_all, hide_all])] + rows)


def new_figure():
    return ipv.figure(width=FIG_WIDTH, height=FIG_HEIGHT)


def finish_figure(name, file_stem):
    ipv.xyzlim(-1, 1)
    ipv.xlabel(f"{name} 1")
    ipv.ylabel(f"{name} 2")
    ipv.zlabel(f"{name} 3")
    if SAVE_HTML:
        FIG_DIR.mkdir(exist_ok=True)
        path = FIG_DIR / f"{file_stem}.html"
        try:
            ipv.save(str(path))
            print(f"  saved {path}")
        except Exception as exc:                                  # a failed save must not stop the run
            print(f"  could not save {path.name}: {exc}")


def show_view(fig, title, side_widgets):
    display(widgets.VBox([widgets.HTML(f"<b>{title}</b>"),
                          widgets.HBox([fig, widgets.VBox(side_widgets)])]))

In [5]:
def plot_full_sampling(E, labels, idx, proj, year, ont, hw):
    """Figure 1: the full 3D embedding. Selected triggers large and coloured by cluster, the rest small."""
    name = PROJECTIONS[proj]
    P = unit_cube(E)
    colors = label_colors(labels)
    rgb = color_array(labels, colors)
    selected = np.zeros(len(E), dtype=bool)
    selected[idx] = True
    noise = labels < 0

    groups = [("selected (the subset)", selected, SIZE_SELECTED),
              ("clustered, not selected", ~selected & ~noise, SIZE_UNSELECTED),
              ("noise, not selected", ~selected & noise, SIZE_NOISE)]

    fig = new_figure()
    items = []
    for text, mask, size in groups:
        if mask.any():
            sc = ipv.scatter(P[mask, 0], P[mask, 1], P[mask, 2], color=rgb[mask], size=size, marker=FULL_MARKER)
            items.append((f"{text} ({int(mask.sum())})", None, sc))
    finish_figure(name, f"TERZ_hw{hw}_ont{ont}_{year}_{proj}3d_sampling_hdbscan_{LABEL_FROM}labels")

    lines = []
    for k in np.unique(labels[labels >= 0]):
        n_k = int((labels == k).sum())
        n_sel = int((labels[idx] == k).sum())
        note = f"{n_sel} selected" if n_k >= MIN_CLUSTER_SIZE else f"discarded (< {MIN_CLUSTER_SIZE})"
        lines.append(f"<div style='white-space:nowrap'><span style='display:inline-block;width:12px;height:12px;"
                     f"margin-right:6px;border-radius:2px;background:{to_hex(colors[k])}'></span>"
                     f"cluster {k}: {n_k} triggers, {note}</div>")
    title = (f"{year} | ont={ont} | hw = {hw} | {name} {N_COMPONENTS}D | HDBSCAN clusters from the {PROJECTIONS[LABEL_FROM]} embedding | "
             f"{len(idx)} of {len(E)} triggers selected")
    show_view(fig, title, [make_controls(items), widgets.HTML("<br>" + "".join(lines))])


def plot_subset_3d(E, labels, proj, year, ont, hw):
    """Figure 2: the 3D embedding of the subset, coloured by the first-round cluster, one tick box per cluster."""
    name = PROJECTIONS[proj]
    P = unit_cube(E)
    colors = label_colors(labels)

    fig = new_figure()
    items = []
    for k in np.unique(labels):
        m = labels == k
        rgb = np.tile(colors[k], (int(m.sum()), 1))
        sc = ipv.scatter(P[m, 0], P[m, 1], P[m, 2], color=rgb, size=SIZE_SUBSET, marker=SUB_MARKER)
        items.append((f"{'noise' if k < 0 else f'cluster {k}'} (n={int(m.sum())})", colors[k], sc))
    finish_figure(name, f"TERZ_hw{hw}_ont{ont}_{year}_{proj}3d_subset_hdbscan_{LABEL_FROM}labels")

    if proj == "tsne":
        params = f"Perplexity = {SUB_PERPLEXITY} | Early Exaggeration = {SUB_EARLY_EXAGGERATION}"
    else:
        params = f"n_neighbors = {SUB_UMAP_N_NEIGHBORS} | min_dist = {SUB_UMAP_MIN_DIST}"
    title = (f"{year} | ont={ont} | hw = {hw} | {name} {N_COMPONENTS}D of the balanced subset | {params} | Metric = {METRIC} | "
             f"{len(labels)} triggers, up to {PER_CLUSTER} per HDBSCAN cluster (labels from {PROJECTIONS[LABEL_FROM]})")
    show_view(fig, title, [make_controls(items)])


def run_full(year, ont, hw):
    """Figure 1 for both projections: where the sampled triggers sit in the full 3D embedding."""
    embeddings = get_embeddings(year, ont, hw)
    idx, labels_full = get_subset(year, ont, hw)
    for proj in PROJECTIONS:
        plot_full_sampling(embeddings[proj], labels_full, idx, proj, year, ont, hw)


def run_subset(year, ont, hw):
    """Figure 2 for both projections: the subset embedded again."""
    idx, labels_full = get_subset(year, ont, hw)
    embeddings = get_subset_embeddings(year, ont, hw, idx)
    for proj in PROJECTIONS:
        plot_subset_3d(embeddings[proj], labels_full[idx], proj, year, ont, hw)

## Run: the sampling in the full 3D embedding

One figure per projection and per `hw`. Each figure is a WebGL scene, so with many `YEARS` x `ONTS` x `HW` the browser gets slow; run fewer cases at a time or call e.g. `run_full(2022, 25, 12)` for a single one.

In [8]:
for year in YEARS:
    for ont in ONTS:
        for hw in HW:
            run_full(year, ont, hw)

2022 ont=25: t-SNE loaded from TERZ_hw6_ont25_2022_tsne3d_perp350_ee14.npy
2022 ont=25: UMAP loaded from TERZ_hw6_ont25_2022_umap3d_nn15_md0.1.npy
2022 ont=25: t-SNE loaded from TERZ_hw6_ont25_2022_tsne3d_perp350_ee14.npy
2022 ont=25: UMAP loaded from TERZ_hw6_ont25_2022_umap3d_nn15_md0.1.npy
2022 ont=25 hw=6: 4 HDBSCAN clusters on the t-SNE 3D embedding, sizes [3204, 1160, 1536, 932], 14419 noise points (not sampled)
  saved TERZ_figures_net_compare_sampling_3d/TERZ_hw6_ont25_2022_tsne3d_sampling_hdbscan_tsnelabels.html


  saved TERZ_figures_net_compare_sampling_3d/TERZ_hw6_ont25_2022_umap3d_sampling_hdbscan_tsnelabels.html


2022 ont=25: t-SNE loaded from TERZ_hw12_ont25_2022_tsne3d_perp350_ee14.npy
2022 ont=25: UMAP loaded from TERZ_hw12_ont25_2022_umap3d_nn15_md0.1.npy
2022 ont=25: t-SNE loaded from TERZ_hw12_ont25_2022_tsne3d_perp350_ee14.npy
2022 ont=25: UMAP loaded from TERZ_hw12_ont25_2022_umap3d_nn15_md0.1.npy
2022 ont=25 hw=12: 2 HDBSCAN clusters on the t-SNE 3D embedding, sizes [5686, 2916], 12649 noise points (not sampled)
  saved TERZ_figures_net_compare_sampling_3d/TERZ_hw12_ont25_2022_tsne3d_sampling_hdbscan_tsnelabels.html


  saved TERZ_figures_net_compare_sampling_3d/TERZ_hw12_ont25_2022_umap3d_sampling_hdbscan_tsnelabels.html


## Run: the subset embedded again in 3D

In [7]:
for year in YEARS:
    for ont in ONTS:
        for hw in HW:
            run_subset(year, ont, hw)

2022 ont=25: t-SNE loaded from TERZ_hw6_ont25_2022_tsne3d_perp350_ee14.npy
2022 ont=25: UMAP loaded from TERZ_hw6_ont25_2022_umap3d_nn15_md0.1.npy
2022 ont=25 hw=6: 4 HDBSCAN clusters on the t-SNE 3D embedding, sizes [3204, 1160, 1536, 932], 14419 noise points (not sampled)
2022 ont=25: 3000 triggers, 1230 features
2022 ont=25 hw=6: computing subset t-SNE (3D) ...
2022 ont=25 hw=6: computing subset UMAP (3D) ...


/home/rauls/Desktop/VirgoBRET/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  saved TERZ_figures_net_compare_sampling_3d/TERZ_hw6_ont25_2022_tsne3d_subset_hdbscan_tsnelabels.html


  saved TERZ_figures_net_compare_sampling_3d/TERZ_hw6_ont25_2022_umap3d_subset_hdbscan_tsnelabels.html


2022 ont=25: t-SNE loaded from TERZ_hw12_ont25_2022_tsne3d_perp350_ee14.npy
2022 ont=25: UMAP loaded from TERZ_hw12_ont25_2022_umap3d_nn15_md0.1.npy
2022 ont=25 hw=12: 2 HDBSCAN clusters on the t-SNE 3D embedding, sizes [5686, 2916], 12649 noise points (not sampled)
2022 ont=25: 1500 triggers, 1230 features
2022 ont=25 hw=12: computing subset t-SNE (3D) ...
2022 ont=25 hw=12: computing subset UMAP (3D) ...


/home/rauls/Desktop/VirgoBRET/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  saved TERZ_figures_net_compare_sampling_3d/TERZ_hw12_ont25_2022_tsne3d_subset_hdbscan_tsnelabels.html


  saved TERZ_figures_net_compare_sampling_3d/TERZ_hw12_ont25_2022_umap3d_subset_hdbscan_tsnelabels.html
